In [2]:
%load_ext autoreload
%autoreload 2

%autosave 30

Autosaving every 30 seconds


In [3]:
import copy

import torch
from torch.utils.data import DataLoader, random_split

from mermaidseg.datasets.dataset import CombinedCoralDataset, CoralNetDataset, MermaidDataset
from mermaidseg.io import get_parser, setup_config, update_config_with_args
from mermaidseg.logger import Logger

In [4]:
import os

os.environ["AWS_PROFILE"] = "mermaid-core"

## CoralNet annotations

Merged CoralNet point annotations are stored as a single Parquet file on S3 (see `CoralNetDataset` in `mermaidseg/datasets/dataset.py`). The block below loads that file with Ibis and inspects the S3 object layout for source `4957` (images are expected under `coralnet-public-images/s4957/images/`).

In [9]:
import boto3
import ibis
import pandas as pd
from great_tables import GT
from ibis import _

# Defaults match CoralNetDataset in mermaidseg/datasets/dataset.py
CORALNET_SOURCE_BUCKET = "dev-datamermaid-sm-sources"
CORALNET_ANNOTATIONS_PARQUET = "coralnet_annotations_30112025.parquet"
CORALNET_ANNOTATIONS_URI = f"s3://{CORALNET_SOURCE_BUCKET}/{CORALNET_ANNOTATIONS_PARQUET}"
CORALNET_SOURCE_S3_PREFIX = "coralnet-public-images"
EXAMPLE_SOURCE_ID = 5086
AUDIT_PATH = "coralnet_source_audit_20260423.parquet"
AUDIT_URI = f"s3://{CORALNET_SOURCE_BUCKET}/dev/{AUDIT_PATH}"
ibis.options.interactive = True

con = ibis.duckdb.connect()
con.raw_sql(
    """
    INSTALL httpfs;
    LOAD httpfs;
    """
)
# Second statement: DuckDB validates the credential chain here — if this fails, fix AWS first
# (skill preflight: `aws sts get-caller-identity`, or rerun the notebook cell that runs check_aws_session).
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
s3_audit = con.read_parquet(AUDIT_URI)

In [12]:
# Recompute is_complete so that it also includes that n_annotations > 0
s3_audit = s3_audit.mutate(
    is_complete=_.has_images_folder & _.has_annotations_csv & _.has_image_list_csv & _.has_annotations_csv & (_.n_annotations > 0)
)

In [14]:
s3_stats = s3_audit.describe().to_pandas()
s3_stats

,name,pos,type,count,nulls,unique,mean,std,min,p25,p50,p75,max
0,has_labelset_csv,5,boolean,1509,0,2,0.597747,NaN,NaN,NaN,NaN,NaN,NaN
1,has_metadata_csv,6,boolean,1509,0,2,0.405567,NaN,NaN,NaN,NaN,NaN,NaN
2,n_images_s3,7,int64,1509,0,502,729.669317,3509.213630,0.0,18.0,50.0,253.0,79062.0
3,n_images_csv,8,int64,1509,0,397,336.215374,1489.577146,0.0,0.0,10.0,104.0,32176.0
4,n_annotations,9,int64,1509,0,492,17768.552684,97664.561643,0.0,0.0,103.0,2310.0,2951200.0
5,n_unique_images_annotated,10,int64,1509,0,381,598.943009,3554.895043,0.0,0.0,3.0,88.0,79064.0
6,n_labels,11,float64,1509,607,134,46.771619,46.662685,1.0,15.0,25.0,75.0,273.0
7,n_metadata_rows,12,float64,1509,897,22,3.485294,3.761478,1.0,1.0,2.0,4.0,32.0
8,annotations_empty,13,boolean,1509,0,2,0.052353,NaN,NaN,NaN,NaN,NaN,NaN
9,image_list_empty,14,boolean,1509,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
summary_data = {
    "Metric": [
        "Total sources",
        "Complete sources",
        "Sources with images folder",
        "Sources with annotations.csv",
        "Sources with image_list.csv",
        "Sources with labelset.csv",
        "Sources with metadata.csv",
        "Sources with image count match",
        "Sources with empty annotations.csv",
        "Sources with empty image_list.csv",
        "Sources with empty labelset.csv",
        "Sources with empty metadata.csv",
    ],
    "Count": [
        s3_audit.count().execute(),
        s3_audit.filter(_.is_complete).count().execute(),
        s3_audit.filter(_.has_images_folder).count().execute(),
        s3_audit.filter(_.has_annotations_csv).count().execute(),
        s3_audit.filter(_.has_image_list_csv).count().execute(),
        s3_audit.filter(_.has_labelset_csv).count().execute(),
        s3_audit.filter(_.has_metadata_csv).count().execute(),
        s3_audit.filter(_.image_count_match).count().execute(),
        s3_audit.filter(_.annotations_empty).count().execute(),
        s3_audit.filter(_.image_list_empty).count().execute(),
        s3_audit.filter(_.labelset_empty).count().execute(),
        s3_audit.filter(_.metadata_empty).count().execute(),
    ],
}
summary_df = pd.DataFrame(summary_data)
total = summary_df.loc[0, "Count"]
summary_df["Percentage"] = (summary_df["Count"] / total * 100).round(1)

In [16]:
GT(summary_df).tab_header(title="CoralNet Source Audit Summary")

GT(_tbl_data=                                Metric  Count  Percentage
0                        Total sources   1509       100.0
1                     Complete sources    812        53.8
2           Sources with images folder   1461        96.8
3         Sources with annotations.csv    898        59.5
4          Sources with image_list.csv    897        59.4
5            Sources with labelset.csv    902        59.8
6            Sources with metadata.csv    612        40.6
7       Sources with image count match    903        59.8
8   Sources with empty annotations.csv     79         5.2
9    Sources with empty image_list.csv      0         0.0
10     Sources with empty labelset.csv      0         0.0
11     Sources with empty metadata.csv      0         0.0, _body=<great_tables._gt_data.Body object at 0x153ce0ef0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='right', column_width=None), ColInfo(var='Percentage', type=<ColInfoTypeEnum.default: 1>, column_label='Percentage', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x13121c140>, _spanners=Spanners([]), _heading=Heading(title='CoralNet Source Audit Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1540cb1d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1540cb470>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1540cb170>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=Tru

In [17]:
totals = s3_audit.aggregate(
    total_images_s3=s3_audit.n_images_s3.sum(),
    total_images_csv=s3_audit.n_images_csv.sum(),
    total_annotations=s3_audit.n_annotations.sum(),
    total_unique_annotated=s3_audit.n_unique_images_annotated.sum(),
    total_labels=s3_audit.n_labels.sum(),
    total_metadata_rows=s3_audit.n_metadata_rows.sum(),
).execute()

totals_df = totals.T.reset_index()
totals_df.columns = ["Metric", "Value"]

In [20]:
GT(totals_df).tab_header(title="Total Resource Counts").fmt_number(columns=["Value"], sep_mark=",", decimals=0)

GT(_tbl_data=                   Metric       Value
0         total_images_s3   1101071.0
1        total_images_csv    507349.0
2       total_annotations  26812746.0
3  total_unique_annotated    903805.0
4            total_labels     42188.0
5     total_metadata_rows      2133.0, _body=<great_tables._gt_data.Body object at 0x153c66420>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x15406f560>, _spanners=Spanners([]), _heading=Heading(title='Total Resource Counts', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1540ca510>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1540c9d60>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1540c8bf0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1539fe5a0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='he

In [7]:
cn_annotations_raw = con.read_parquet(CORALNET_ANNOTATIONS_URI)
cn_for_source = cn_annotations_raw.filter(cn_annotations_raw.source_id == EXAMPLE_SOURCE_ID)

anno_summary = cn_for_source.aggregate(
    n_points=cn_for_source.count(),
    n_images=cn_for_source.image_id.nunique(),
).execute()

In [8]:
cn_annotations_raw

┏━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┓
┃ source_id ┃ image_id ┃ row   ┃ col   ┃ coralnet_id ┃
┡━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━┩
│ int64     │ string   │ int64 │ int64 │ int64       │
├───────────┼──────────┼───────┼───────┼─────────────┤
│        23 │ 12805    │   735 │  1008 │         112 │
│        23 │ 12805    │   663 │  1682 │         106 │
│        23 │ 12805    │   955 │  1737 │         106 │
│        23 │ 12805    │  1034 │  1431 │         105 │
│        23 │ 12805    │   851 │  2036 │         106 │
│        23 │ 12805    │   833 │  2038 │         100 │
│        23 │ 12805    │  1235 │  1158 │         111 │
│        23 │ 12805    │  1294 │  1312 │         105 │
│        23 │ 12805    │  1194 │  1319 │         105 │
│        23 │ 12805    │  1498 │  3096 │         105 │
│         … │ …        │     … │     … │           … │
└───────────┴──────────┴───────┴───────┴─────────────┘

In [11]:
anno_summary

,n_points,n_images
0,20200,1000


In [7]:
sources_p = cn_annotations_raw.select(_.source_id).distinct()

In [ ]:
LIST_OF_SOURCES = [4957, 5086, 5444, 5847, 6521, 6537, 6687, 6699]

In [ ]:
sources_p.filter(sources_p.source_id.isin(LIST_OF_SOURCES)).execute()

In [8]:
s3 = boto3.client("s3")


def list_prefix_layout(bucket: str, prefix: str) -> tuple[list[str], list[str]]:
    """Return common prefix 'folders' and a few object keys directly under `prefix`."""
    out = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/", MaxKeys=1000)
    prefixes = [p["Prefix"] for p in out.get("CommonPrefixes", []) or []]
    keys = [o["Key"] for o in out.get("Contents", []) or [] if o["Key"] != prefix]
    return prefixes, keys


source_root = f"{CORALNET_SOURCE_S3_PREFIX}/s{EXAMPLE_SOURCE_ID}/"
prefixes_l1, keys_l1 = list_prefix_layout(CORALNET_SOURCE_BUCKET, source_root)
images_prefix = f"{source_root}images/"
prefixes_images, keys_images = list_prefix_layout(CORALNET_SOURCE_BUCKET, images_prefix)

sample_rows = cn_for_source.limit(5).execute()

In [14]:
keys_l1

['coralnet-public-images/s5086/annotations.csv',
 'coralnet-public-images/s5086/image_list.csv',
 'coralnet-public-images/s5086/labelset.csv',
 'coralnet-public-images/s5086/metadata.csv']

### Download CoralNet source CSVs locally

Keys from `keys_l1` (e.g. `coralnet-public-images/s4957/*.csv`) are mirrored under `coralnet_s3_mirror/` with the same key paths.

Run after the cell that defines `s3`, `CORALNET_SOURCE_BUCKET`, `source_root`, and `keys_l1`.

In [ ]:
from pathlib import Path

from tqdm.auto import tqdm

LOCAL_CORALNET_MIRROR = Path("coralnet_s3_mirror")


def download_s3_keys_preserve_layout(
    bucket: str,
    keys: list[str],
    dest_root: Path,
    *,
    show_progress: bool = False,
) -> list[Path]:
    dest_root = dest_root.resolve()
    out_paths: list[Path] = []
    it = keys
    if show_progress:
        it = tqdm(keys, desc="S3 download")
    for key in it:
        local = dest_root / key
        local.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(local))
        out_paths.append(local)
    return out_paths


csv_keys = [k for k in keys_l1 if k.endswith(".csv")]
if not csv_keys:
    csv_keys = [
        f"{source_root}{name}"
        for name in (
            "annotations.csv",
            "image_list.csv",
            "labelset.csv",
            "metadata.csv",
        )
    ]

downloaded_paths = download_s3_keys_preserve_layout(
    CORALNET_SOURCE_BUCKET,
    csv_keys,
    LOCAL_CORALNET_MIRROR,
)

In [ ]:
from IPython.display import display

display(
    GT(pd.DataFrame({"local_path": [str(p) for p in downloaded_paths]})).tab_header(
        title="Downloaded CoralNet CSVs",
        subtitle=f"Mirror root: {LOCAL_CORALNET_MIRROR.resolve()}",
    )
)

### Images: collect file metadata from local mirror

Scans the local `coralnet_s3_mirror/.../s{EXAMPLE_SOURCE_ID}/images/` directory, collects metadata (file size, mtime, width/height from Pillow) for every image, then writes `images/image_file_metadata.parquet` (falls back to `.csv` if Parquet is unavailable). Run this after images have been downloaded to the mirror.

In [ ]:
from PIL import Image

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".bmp", ".gif")

LOCAL_IMAGES_DIR = LOCAL_CORALNET_MIRROR / CORALNET_SOURCE_S3_PREFIX / f"s{EXAMPLE_SOURCE_ID}" / "images"


def collect_image_metadata_from_local_dir(
    images_dir: Path,
    mirror_root: Path,
) -> pd.DataFrame:
    mirror_root = mirror_root.resolve()
    images_dir = images_dir.resolve()
    records: list[dict] = []
    if not images_dir.exists():
        return pd.DataFrame.from_records(records)

    for img_path in sorted(images_dir.rglob("*")):
        if not img_path.is_file():
            continue
        if not any(img_path.name.lower().endswith(ext) for ext in IMAGE_EXTENSIONS):
            continue

        img_path_resolved = img_path.resolve()
        st = img_path_resolved.stat()
        s3_key = str(img_path_resolved.relative_to(mirror_root))
        width = height = None
        pil_format = pil_mode = probe_error = None
        try:
            with Image.open(img_path_resolved) as im:
                width, height = im.size
                pil_format = im.format
                pil_mode = im.mode
        except Exception as exc:  # noqa: BLE001
            probe_error = repr(exc)

        records.append(
            {
                "s3_key": s3_key,
                "local_path": str(img_path_resolved),
                "bytes_on_disk": st.st_size,
                "mtime_ns": getattr(st, "st_mtime_ns", int(st.st_mtime * 1e9)),
                "width_px": width,
                "height_px": height,
                "pil_format": pil_format,
                "pil_mode": pil_mode,
                "probe_error": probe_error,
            }
        )
    return pd.DataFrame.from_records(records)

In [ ]:
image_metadata_df = collect_image_metadata_from_local_dir(
    LOCAL_IMAGES_DIR,
    LOCAL_CORALNET_MIRROR,
)

# 1. Setup

In [23]:
device_count = torch.cuda.device_count()
for i in range(device_count):
    print(f"CUDA Device {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

## Config

#
#
#
 
P
e
r
-
s
o
u
r
c
e
 
C
S
V
 
i
n
g
e
s
t
 
a
n
d
 
p
o
i
n
t
 
r
e
s
c
a
l
i
n
g


N
o
t
e
b
o
o
k
 
l
a
y
o
u
t
:
 
h
e
l
p
e
r
 
`
d
e
f
`
 
c
e
l
l
s
 
a
r
e
 
s
e
p
a
r
a
t
e
 
f
r
o
m
 
t
h
e
 
c
e
l
l
s
 
t
h
a
t
 
c
a
l
l
 
t
h
e
m
 
(
[
`
n
o
t
e
b
o
o
k
-
d
e
f
i
n
e
-
t
h
e
n
-
r
u
n
-
c
e
l
l
s
.
m
d
c
`
]
(
.
.
/
.
c
u
r
s
o
r
/
r
u
l
e
s
/
n
o
t
e
b
o
o
k
-
d
e
f
i
n
e
-
t
h
e
n
-
r
u
n
-
c
e
l
l
s
.
m
d
c
)
)
.
 
R
u
n
 
p
a
t
h
 
i
s
 
s
p
l
i
t
:
 
l
o
a
d
 
o
n
e
 
p
i
n
n
e
d
 
i
m
a
g
e
 
→
 
p
r
i
n
t
/
t
a
b
l
e
 
+
 
s
h
o
w
 
n
a
t
i
v
e
 
p
r
e
v
i
e
w
 
→
 
r
e
s
i
z
e
/
a
s
s
e
r
t
s
/
p
l
o
t
s
 
i
n
 
l
a
t
e
r
 
c
e
l
l
s
.


*
*
S
i
n
g
l
e
-
i
m
a
g
e
 
t
e
s
t
:
*
*
 
`
D
E
M
O
_
I
M
A
G
E
_
S
T
E
M
`
 
d
e
f
a
u
l
t
s
 
t
o
 
o
n
e
 
f
i
l
e
n
a
m
e
 
s
t
e
m
 
u
n
d
e
r
 
`
L
O
C
A
L
_
I
M
A
G
E
S
_
D
I
R
`
;
 
r
u
n
 
c
e
l
l
s
 
p
r
i
n
t
 
t
h
a
t
 
p
a
t
h
 
a
n
d
 
s
h
o
w
 
t
h
e
 
n
a
t
i
v
e
 
R
G
B
 
b
e
f
o
r
e
 
r
e
s
i
z
e
/
p
a
t
c
h
 
c
h
e
c
k
s
.
 
S
e
t
 
i
t
 
t
o
 
`
N
o
n
e
`
 
t
o
 
a
u
t
o
-
p
i
c
k
.


S
e
e
 
[
d
o
c
s
/
c
o
r
a
l
n
e
t
-
d
a
t
a
-
s
o
u
r
c
e
s
.
m
d
]
(
.
.
/
d
o
c
s
/
c
o
r
a
l
n
e
t
-
d
a
t
a
-
s
o
u
r
c
e
s
.
m
d
)
 
f
o
r
 
S
3
 
l
a
y
o
u
t
,
 
C
S
V
 
s
c
h
e
m
a
s
,
 
a
n
d
 
r
e
c
o
m
m
e
n
d
e
d
 
*
*
7
6
8
×
5
1
2
*
*
 
(
3
:
2
)
 
r
e
s
i
z
e
 
w
i
t
h
 
u
n
i
f
o
r
m
 
p
o
i
n
t
 
s
c
a
l
i
n
g
.


*
*
T
r
a
i
n
i
n
g
 
c
o
d
e
 
p
a
t
h
 
(
d
i
f
f
e
r
e
n
t
 
f
r
o
m
 
t
h
i
s
 
s
e
c
t
i
o
n
)
:
*
*
 
`
B
a
s
e
C
o
r
a
l
D
a
t
a
s
e
t
`
 
i
n
 
[
`
m
e
r
m
a
i
d
s
e
g
/
d
a
t
a
s
e
t
s
/
d
a
t
a
s
e
t
.
p
y
`
]
(
.
.
/
m
e
r
m
a
i
d
s
e
g
/
d
a
t
a
s
e
t
s
/
d
a
t
a
s
e
t
.
p
y
)
 
r
a
s
t
e
r
i
z
e
s
 
`
r
o
w
`
 
/
 
`
c
o
l
`
 
w
i
t
h
 
`
c
r
e
a
t
e
_
a
n
n
o
t
a
t
i
o
n
_
m
a
s
k
`
 
a
t
 
f
u
l
l
 
i
m
a
g
e
 
r
e
s
o
l
u
t
i
o
n
,
 
t
h
e
n
 
a
p
p
l
i
e
s
 
A
l
b
u
m
e
n
t
a
t
i
o
n
s
 
t
o
 
*
*
`
i
m
a
g
e
`
 
a
n
d
 
`
m
a
s
k
`
 
t
o
g
e
t
h
e
r
*
*
—
a
n
n
o
t
a
t
i
o
n
 
r
o
w
s
 
a
r
e
 
n
o
t
 
t
r
a
n
s
f
o
r
m
e
d
 
a
s
 
k
e
y
p
o
i
n
t
 
a
r
r
a
y
s
.
 
T
h
e
 
c
e
l
l
s
 
b
e
l
o
w
 
d
e
m
o
n
s
t
r
a
t
e
 
e
x
p
l
i
c
i
t
 
`
(
c
o
l
,
 
r
o
w
)
`
 
r
e
s
c
a
l
i
n
g
 
f
o
r
 
a
u
d
i
t
s
 
a
n
d
 
p
a
r
i
t
y
 
w
i
t
h
 
t
h
e
 
d
o
c
;
 
t
h
e
y
 
c
o
m
p
l
e
m
e
n
t
 
(
d
o
 
n
o
t
 
r
e
p
l
a
c
e
)
 
t
h
e
 
m
a
s
k
-
b
a
s
e
d
 
t
r
a
i
n
i
n
g
 
p
i
p
e
l
i
n
e
.



In [24]:
import cv2
import numpy as np

In [25]:
LOCAL_SOURCE_DIR = LOCAL_CORALNET_MIRROR / CORALNET_SOURCE_S3_PREFIX / f"s{EXAMPLE_SOURCE_ID}"
ANN_PATH = LOCAL_SOURCE_DIR / "annotations.csv"
IMG_LIST_PATH = LOCAL_SOURCE_DIR / "image_list.csv"
LABELSET_PATH = LOCAL_SOURCE_DIR / "labelset.csv"
META_PATH = LOCAL_SOURCE_DIR / "metadata.csv"

if not LOCAL_SOURCE_DIR.is_dir():
    raise FileNotFoundError(f"Expected mirror directory missing: {LOCAL_SOURCE_DIR.resolve()}. Run the download cells above.")

annotations_df = pd.read_csv(ANN_PATH)
image_list_df = pd.read_csv(IMG_LIST_PATH)
labelset_df = pd.read_csv(LABELSET_PATH)
metadata_df = pd.read_csv(META_PATH)

annotations_labeled = annotations_df.merge(
    labelset_df,
    on="Label ID",
    how="left",
    suffixes=("", "_labelset"),
)

summary = pd.DataFrame(
    {
        "table": ["annotations", "image_list", "labelset", "metadata", "annotations_labeled"],
        "n_rows": [
            len(annotations_df),
            len(image_list_df),
            len(labelset_df),
            len(metadata_df),
            len(annotations_labeled),
        ],
    }
)
display(GT(summary).tab_header(title="Per-source CSV ingest", subtitle=str(LOCAL_SOURCE_DIR.resolve())))

GT(_tbl_data=                 table  n_rows
0          annotations   78170
1           image_list     634
2             labelset     112
3             metadata       4
4  annotations_labeled   78170, _body=<great_tables._gt_data.Body object at 0x1b03ae3c0>, _boxhead=Boxhead([ColInfo(var='table', type=<ColInfoTypeEnum.default: 1>, column_label='table', column_align='left', column_width=None), ColInfo(var='n_rows', type=<ColInfoTypeEnum.default: 1>, column_label='n_rows', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1b03ae900>, _spanners=Spanners([]), _heading=Heading(title='Per-source CSV ingest', subtitle='/Users/laurenyee/Projects/mermaid-segmentation/.worktrees/feat/sagemaker-jupyter-resilience/nbs/coralnet_s3_mirror/coralnet-public-images/s4957', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1b00a3380>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1b00a1eb0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1b00a26c0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, categor

In [26]:
from mermaidseg.datasets.utils import create_annotation_mask

TARGET_W, TARGET_H = 768, 512
PATCH_SIZE = 16


def resize_image_with_points(
    image: np.ndarray,
    points_xy: np.ndarray,
    target_size: tuple[int, int],
    interpolation: int = cv2.INTER_AREA,
) -> tuple[np.ndarray, np.ndarray]:
    """Resize image; rescale points as (col, row). OpenCV ``target_size`` is (width, height)."""
    h_orig, w_orig = image.shape[:2]
    w_new, h_new = target_size
    orig_aspect = w_orig / h_orig
    target_aspect = w_new / h_new
    if abs(orig_aspect - target_aspect) > 0.01:
        msg = f"Aspect ratio mismatch: {orig_aspect:.4f} vs {target_aspect:.4f}."
        raise ValueError(msg)
    scale = w_new / w_orig
    image_resized = cv2.resize(image, target_size, interpolation=interpolation)
    pts = points_xy * scale
    return image_resized, pts


def validate_points(points_xy: np.ndarray, image_shape: tuple[int, ...]) -> np.ndarray:
    h, w = image_shape[:2]
    m = (points_xy[:, 0] >= 0) & (points_xy[:, 0] < w) & (points_xy[:, 1] >= 0) & (points_xy[:, 1] < h)
    return points_xy[m]


def map_point_to_patch_grid(
    point_col: float,
    point_row: float,
    native_width: int,
    native_height: int,
    target_width: int = TARGET_W,
    target_height: int = TARGET_H,
    patch_size: int = PATCH_SIZE,
) -> tuple[int, int]:
    sx = target_width / native_width
    sy = target_height / native_height
    sc = point_col * sx
    sr = point_row * sy
    pc = int(sc // patch_size)
    pr = int(sr // patch_size)
    pc = min(pc, (target_width // patch_size) - 1)
    pr = min(pr, (target_height // patch_size) - 1)
    return pc, pr

In [27]:
# Pin exactly one image by stem (e.g. "4165480" → 4165480.jpg under LOCAL_IMAGES_DIR).
# Default: single-file smoke test. Set to None to auto-pick the first usable match.
DEMO_IMAGE_STEM: str | None = "4165480"

### Build Name-to-image_id mapping

CoralNet CSVs use the original filename in the "Name" column (e.g., `2023-10-19_FSMATOLLS_SAPW-2_T1_10.JPG`), but local/S3 images are stored by their CoralNet image ID (e.g., `4165480.jpg`). This section extracts the mapping from `image_list.csv` using regex on the `Image Page` column (`/image/{id}/view/`).

In [30]:
import re

# Regex to extract image_id from Image Page path: /image/5724823/view/ → 5724823
PAGE_ID_PATTERN = re.compile(r"/image/(\d+)/view/")

# Alternative: from Image URL media/images/yo58x1ak8n.JPG → yo58x1ak8n
URL_ID_PATTERN = re.compile(r"media/images/([a-zA-Z0-9]+)\.jpe?g", re.IGNORECASE)


def extract_image_id_from_page(image_page: str) -> str | None:
    """Extract numeric image_id from Image Page path like /image/5724823/view/."""
    match = PAGE_ID_PATTERN.search(image_page)
    if match:
        return match.group(1)
    return None


def extract_image_id_from_url(image_url: str) -> str | None:
    """Extract image_id from Image URL media/images/{id}.JPG."""
    match = URL_ID_PATTERN.search(image_url)
    if match:
        return match.group(1)
    return None


# Build mapping dataframe from image_list_df (loaded in cell above)
mapping_df = image_list_df.copy()
mapping_df["image_id_from_page"] = mapping_df["Image Page"].apply(extract_image_id_from_page)
mapping_df["image_id_from_url"] = mapping_df["Image URL"].apply(extract_image_id_from_url)

# Prefer page-derived ID (numeric) as canonical; fall back to URL-derived if needed
mapping_df["image_id"] = mapping_df["image_id_from_page"].fillna(mapping_df["image_id_from_url"])

# Clean original Name (remove " - Unconfirmed" / " - Confirmed" suffixes)
mapping_df["original_name_clean"] = mapping_df["Name"].str.replace(r" - (Unconfirmed|Confirmed)$", "", regex=True)

# Check which images exist locally
local_image_stems = {f.stem for f in LOCAL_IMAGES_DIR.glob("*.jpg")}
mapping_df["has_local_file"] = mapping_df["image_id"].isin(local_image_stems)

In [31]:
# Summary table of the mapping
summary_df = pd.DataFrame(
    {
        "metric": [
            "total_image_list_entries",
            "with_extracted_image_id",
            "missing_image_id",
            "with_local_file",
            "missing_local_file",
        ],
        "count": [
            len(mapping_df),
            mapping_df["image_id"].notna().sum(),
            mapping_df["image_id"].isna().sum(),
            mapping_df["has_local_file"].sum(),
            (~mapping_df["has_local_file"]).sum(),
        ],
    }
)

display(
    GT(summary_df).tab_header(
        title=f"Name-to-image_id mapping summary (source {EXAMPLE_SOURCE_ID})", subtitle=f"Local images dir: {LOCAL_IMAGES_DIR.resolve()}"
    )
)

# Sample of the mapping
sample_cols = ["original_name_clean", "image_id", "has_local_file", "Image Page"]
display(GT(mapping_df[sample_cols].head(10)).tab_header(title="Sample mapping rows"))

GT(_tbl_data=                     metric  count
0  total_image_list_entries    634
1   with_extracted_image_id    634
2          missing_image_id      0
3           with_local_file      0
4        missing_local_file    634, _body=<great_tables._gt_data.Body object at 0x1b00a34a0>, _boxhead=Boxhead([ColInfo(var='metric', type=<ColInfoTypeEnum.default: 1>, column_label='metric', column_align='left', column_width=None), ColInfo(var='count', type=<ColInfoTypeEnum.default: 1>, column_label='count', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1b0479730>, _spanners=Spanners([]), _heading=Heading(title='Name-to-image_id mapping summary (source 4957)', subtitle='Local images dir: /Users/laurenyee/Projects/mermaid-segmentation/.worktrees/feat/sagemaker-jupyter-resilience/nbs/coralnet_s3_mirror/coralnet-public-images/s4957/images', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1b03cf1d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1b03cea50>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1b03cee40>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading

GT(_tbl_data=          original_name_clean image_id  has_local_file            Image Page
0  2025-8-12_PNI-17_T5_25.JPG  5724823           False  /image/5724823/view/
1  2025-8-12_PNI-17_T5_26.JPG  5724826           False  /image/5724826/view/
2  2025-8-12_PNI-17_T5_27.JPG  5724832           False  /image/5724832/view/
3  2025-8-12_PNI-17_T5_28.JPG  5724835           False  /image/5724835/view/
4  2025-8-12_PNI-17_T5_29.JPG  5724840           False  /image/5724840/view/
5   2025-8-12_PNI-17_T5_2.JPG  5724744           False  /image/5724744/view/
6  2025-8-12_PNI-17_T5_30.JPG  5724843           False  /image/5724843/view/
7  2025-8-12_PNI-17_T5_31.JPG  5724846           False  /image/5724846/view/
8  2025-8-12_PNI-17_T5_32.JPG  5724849           False  /image/5724849/view/
9  2025-8-12_PNI-17_T5_33.JPG  5724852           False  /image/5724852/view/, _body=<great_tables._gt_data.Body object at 0x1b03cfaa0>, _boxhead=Boxhead([ColInfo(var='original_name_clean', type=<ColInfoTypeEnum.default: 1>, column_label='original_name_clean', column_align='left', column_width=None), ColInfo(var='image_id', type=<ColInfoTypeEnum.default: 1>, column_label='image_id', column_align='right', column_width=None), ColInfo(var='has_local_file', type=<ColInfoTypeEnum.default: 1>, column_label='has_local_file', column_align='center', column_width=None), ColInfo(var='Image Page', type=<ColInfoTypeEnum.default: 1>, column_label='Image Page', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1b0387b30>, _spanners=Spanners([]), _heading=Heading(title='Sample mapping rows', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1b0479e20>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1b0479eb0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1b0479c10>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='

In [ ]:
# Save the mapping CSV for the ~1086 images (or whatever is in image_list)
MAPPING_OUTPUT_PATH = LOCAL_SOURCE_DIR / "name_to_image_id_mapping.csv"

output_df = mapping_df[
    [
        "original_name_clean",
        "image_id",
        "has_local_file",
        "Name",  # original with suffixes
        "Image Page",
        "Image URL",
    ]
].copy()

output_df.to_csv(MAPPING_OUTPUT_PATH, index=False)
print(f"Saved mapping CSV: {MAPPING_OUTPUT_PATH.resolve()}")
print(f"  Total rows: {len(output_df)}")
print(f"  With local file: {output_df['has_local_file'].sum()}")
print(f"  Without local file: {(~output_df['has_local_file']).sum()}")

In [28]:
def _execution_to_pandas(obj: object) -> pd.DataFrame:
    if isinstance(obj, pd.DataFrame):
        return obj
    if hasattr(obj, "to_pandas"):
        return obj.to_pandas()
    return pd.DataFrame(obj)


def _parquet_rows_for_image_id(pid: str) -> pd.DataFrame:
    """Rows for ``pid`` (stem of ``{image_id}.jpg``) from merged Parquet, str or int ``image_id``."""
    pid = str(pid).strip()
    pid_int: int | None = int(pid) if pid.isdigit() else None

    pq = _execution_to_pandas(cn_for_source.execute())
    if len(pq) and "image_id" in pq.columns:
        key = pq["image_id"].astype(str).str.strip()
        m = key == pid
        if not m.any() and pid_int is not None:
            m = pq["image_id"] == pid_int
        if m.any():
            return pq.loc[m]

    tbl = cn_annotations_raw
    for val in (pid, pid_int):
        if val is None:
            continue
        try:
            sel = tbl.filter((tbl.source_id == EXAMPLE_SOURCE_ID) & (tbl.image_id == val))
            out = _execution_to_pandas(sel.execute())
            if len(out):
                return out
        except (AttributeError, TypeError, ValueError, ibis.IbisError):
            continue
    return pd.DataFrame()


def _points_for_stem(stem: str) -> tuple[np.ndarray, str]:
    """Annotation points (N×2 col,row) and a short source label."""
    stem = str(stem).strip()
    sub = _parquet_rows_for_image_id(stem)
    if len(sub):
        return sub[["col", "row"]].to_numpy(dtype=np.float64), "parquet"
    n = annotations_df["Name"].astype(str)
    m = (n == f"{stem}.jpg") | (n == stem) | (n.str.lower() == f"{stem}.jpg".lower())
    grp = annotations_df.loc[m]
    if len(grp):
        return grp[["Column", "Row"]].to_numpy(dtype=np.float64), "annotations.csv"
    return np.zeros((0, 2)), "none"


def pick_demo_image_and_points(
    *,
    demo_stem: str | None = None,
) -> tuple[np.ndarray, np.ndarray, str, Path]:
    """Return (RGB uint8 HxWx3, points Nx2 as col,row, provenance string, absolute image path).

    If ``demo_stem`` is set, only that filename stem (under ``LOCAL_IMAGES_DIR``) is used.
    Otherwise ``DEMO_IMAGE_STEM`` is used when non-empty; if both are unset, pick the first
    usable match (CSV ``Name`` or sorted ``*.jpg`` / ``*.jpeg`` with points).
    """
    img_dir = LOCAL_IMAGES_DIR
    stem_req = demo_stem if demo_stem is not None else DEMO_IMAGE_STEM
    if stem_req is not None:
        stem_req = str(stem_req).strip() or None

    if stem_req is not None:
        path: Path | None = None
        for ext in (".jpg", ".jpeg"):
            cand = img_dir / f"{stem_req}{ext}"
            if cand.is_file():
                path = cand
                break
        if path is None:
            raise FileNotFoundError(f"No image file for stem {stem_req!r} under {img_dir.resolve()}")
        pts, src = _points_for_stem(stem_req)
        if pts.shape[0] == 0:
            raise FileNotFoundError(f"No annotation rows for stem {stem_req!r} (Parquet + annotations.csv)")
        bgr = cv2.imread(str(path))
        if bgr is None:
            raise FileNotFoundError(f"OpenCV could not read {path}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        prov = f"{src} + {path.name} (pinned stem)"
        return rgb, pts, prov, path.resolve()

    lower_names: dict[str, Path] | None = None
    if img_dir.is_dir():
        lower_names = {f.name.lower(): f for f in img_dir.iterdir() if f.is_file()}

    if img_dir.is_dir() and lower_names is not None:
        for name in annotations_df["Name"].unique():
            cand = img_dir / str(name)
            if cand.is_file():
                grp = annotations_df.loc[annotations_df["Name"] == name]
                pts = grp[["Column", "Row"]].to_numpy(dtype=np.float64)
                bgr = cv2.imread(str(cand))
                if bgr is None:
                    continue
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                return rgb, pts, f"annotations.csv Name + file: {name}", cand.resolve()
            if str(name).lower() in lower_names:
                path = lower_names[str(name).lower()]
                grp = annotations_df.loc[annotations_df["Name"] == name]
                pts = grp[["Column", "Row"]].to_numpy(dtype=np.float64)
                bgr = cv2.imread(str(path))
                if bgr is None:
                    continue
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                return (
                    rgb,
                    pts,
                    f"annotations.csv Name + file (case-insensitive): {path.name}",
                    path.resolve(),
                )

    if img_dir.is_dir():
        seen: set[Path] = set()
        for path in sorted(img_dir.iterdir()):
            if not path.is_file() or path in seen:
                continue
            if path.suffix.lower() not in (".jpg", ".jpeg"):
                continue
            seen.add(path)
            pid = path.stem.strip()
            sub = _parquet_rows_for_image_id(pid)
            if len(sub) == 0:
                continue
            pts = sub[["col", "row"]].to_numpy(dtype=np.float64)
            bgr = cv2.imread(str(path))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            return (
                rgb,
                pts,
                f"Parquet + local file {path.name} (image_id={pid})",
                path.resolve(),
            )

    raise FileNotFoundError(
        "No demo image: need ``images/*.jpg`` under the mirror plus Parquet rows for that ``image_id`` (merged Parquet), "
        "or a CSV ``Name`` that matches a filename. If Parquet has no rows for this ``source_id``, run the Ibis cells above "
        "and confirm S3 access to ``coralnet_annotations_*.parquet``. Set ``DEMO_IMAGE_STEM`` to a known stem to test one file."
    )

In [29]:
from IPython.display import display

demo_rgb, demo_points_xy, demo_source, demo_path = pick_demo_image_and_points()

display(
    GT(
        pd.DataFrame(
            {
                "filename": [demo_path.name],
                "local_path": [str(demo_path)],
                "provenance": [demo_source],
                "native_hw": [demo_rgb.shape[:2]],
                "n_points": [len(demo_points_xy)],
            }
        )
    ).tab_header(
        title="Pinned demo image (single-image test)",
        subtitle="Change DEMO_IMAGE_STEM or set it to None to auto-pick",
    )
)

FileNotFoundError: No annotation rows for stem '4165480' (Parquet + annotations.csv)

In [ ]:
# Print the image being worked on (separate from selection for visibility)
print(
    "Working on this image:\n"
    f"  file: {demo_path.name}\n"
    f"  path: {demo_path}\n"
    f"  source: {demo_source}\n"
    f"  shape (H, W, C): {demo_rgb.shape}\n"
    f"  annotation points (rows): {len(demo_points_xy)}"
)

In [ ]:
# Native RGB preview for the pinned file (compute vs plot: separate cell)
try:
    import holoviews as hv

    hv.extension("bokeh")
    _h0, _w0 = demo_rgb.shape[:2]
    _mw = min(960, _w0)
    hv.RGB(demo_rgb, bounds=(0, 0, _w0, _h0)).opts(
        invert_yaxis=True,
        title=f"Native image — {demo_path.name}",
        width=int(_mw),
        height=int(_mw * _h0 / _w0),
    )
except ModuleNotFoundError:
    import matplotlib.pyplot as plt

    _fig, _ax = plt.subplots(figsize=(12, 8))
    _ax.imshow(demo_rgb)
    _ax.set_title(f"Native image — {demo_path.name}")
    _ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Compute: resize and prepare mask (no asserts)
h0, w0 = demo_rgb.shape[:2]
demo_points_xy = validate_points(demo_points_xy, demo_rgb.shape)

rgb_resized, pts_resized = resize_image_with_points(
    demo_rgb,
    demo_points_xy,
    (TARGET_W, TARGET_H),
    interpolation=cv2.INTER_AREA,
)
scale = TARGET_W / w0

# ``create_annotation_mask`` smoke: same geometry as training (native full-res mask).
_lab = "demo_point"
_toy = pd.DataFrame(
    {
        "row": np.clip(demo_points_xy[:, 1].round().astype(np.int64), 0, h0 - 1),
        "col": np.clip(demo_points_xy[:, 0].round().astype(np.int64), 0, w0 - 1),
        "benthic_attribute_name": [_lab] * len(demo_points_xy),
    }
)
_mask_native = create_annotation_mask(_toy, demo_rgb.shape, {_lab: 1}, padding=None)

In [ ]:
# Asserts and tests (separate from computation per define-then-run rule)
assert len(demo_points_xy) > 0, "no in-bounds annotation points for demo image"
assert np.isclose(scale, TARGET_H / h0), "uniform scale for 3:2"
assert np.allclose(scale, TARGET_W / float(w0))

_pts_ok = validate_points(pts_resized, rgb_resized.shape)
assert _pts_ok.shape[0] == demo_points_xy.shape[0]
for i in range(min(5, len(demo_points_xy))):
    pc, pr = map_point_to_patch_grid(
        float(demo_points_xy[i, 0]),
        float(demo_points_xy[i, 1]),
        native_width=w0,
        native_height=h0,
    )
    assert 0 <= pc < TARGET_W // PATCH_SIZE
    assert 0 <= pr < TARGET_H // PATCH_SIZE

assert _mask_native.shape[:2] == (h0, w0)

print(
    "Resize + point tests passed.",
    {"demo_source": demo_source, "native_hw": (h0, w0), "n_points": len(demo_points_xy)},
)

In [ ]:
# Subsample for plotting (avoid huge Bokeh payloads)
N_VIS = min(200, len(pts_resized))
rng = np.linspace(0, len(pts_resized) - 1, num=N_VIS, dtype=int)
pts_vis = pts_resized[rng]

In [ ]:
try:
    import holoviews as hv

    hv.extension("bokeh")
    _hr, _wr = rgb_resized.shape[:2]
    _under = hv.RGB(rgb_resized, bounds=(0, 0, _wr, _hr)).opts(
        invert_yaxis=True,
        title=f"768×512 resize + points (n={len(pts_vis)}) — {demo_path.name}",
    )
    _overlay = hv.Points(pts_vis, kdims=["col", "row"]).opts(color="red", size=5, alpha=0.65)
    _under * _overlay
except ModuleNotFoundError:
    import matplotlib.pyplot as plt

    _fig, _ax = plt.subplots(figsize=(12, 8))
    _ax.imshow(rgb_resized)
    _ax.scatter(pts_vis[:, 0], pts_vis[:, 1], s=6, c="red", alpha=0.6)
    _ax.set_title(f"768×512 resize + points (n={len(pts_vis)}) — {demo_path.name}")
    plt.tight_layout()
    plt.show()

In [ ]:
cfg = setup_config(
    config_path="../configs/linear-dinov3-base.yaml",
    config_base_path="../configs/combined_mermaid_coralnet.yaml",
)

args_input = "--run-name=combined_base_dinov3 --log-epochs=5"
args_input = args_input.split(" ")

parser = get_parser()
args = parser.parse_args(args_input)

cfg = update_config_with_args(cfg, args)
cfg_logger = copy.deepcopy(cfg)

# 2. Dataset

In [ ]:
transforms = {}
for split in cfg.augmentation:
    transforms[split] = A.Compose(
        [getattr(A, transform_name)(**transform_params) for transform_name, transform_params in cfg.augmentation[split].items()]
    )

In [ ]:
batch_size = cfg.data.pop("batch_size", 8)
class_subset = cfg.data.class_subset
padding = cfg.data.padding

In [ ]:
mermaid_full = MermaidDataset(
    transform=transforms["train"],
    class_subset=class_subset,
    padding=padding,
)

total_size = len(mermaid_full)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(seed)
mermaid_train, mermaid_val, mermaid_test = random_split(mermaid_full, [train_size, val_size, test_size], generator=generator)

print(f"MERMAID splits: train={train_size}, val={val_size}, test={test_size}")

In [ ]:
def _load_sources(path: str) -> list:
    return pd.read_csv(path, header=0).to_numpy().flatten().tolist()


coralnet_train = CoralNetDataset(
    transform=transforms["train"],
    class_subset=class_subset,
    padding=padding,
    whitelist_sources=_load_sources("../configs/focus_sources_train.csv"),
)
coralnet_val = CoralNetDataset(
    transform=transforms["val"],
    class_subset=class_subset,
    padding=padding,
    whitelist_sources=_load_sources("../configs/focus_sources_val.csv"),
)
coralnet_test = CoralNetDataset(
    transform=transforms["test"],
    class_subset=class_subset,
    padding=padding,
    whitelist_sources=_load_sources("../configs/focus_sources_test.csv"),
)

print(f"CoralNet splits: train={len(coralnet_train)}, val={len(coralnet_val)}, test={len(coralnet_test)}")

In [ ]:
MERMAID_CAP = 8000
CORALNET_CAP = 8000


def _subsample(dataset, cap, generator):
    if len(dataset) <= cap:
        return dataset
    return random_split(dataset, [cap, len(dataset) - cap], generator=generator)[0]


sub_generator = torch.Generator().manual_seed(seed)

# Subsample MERMAID splits proportionally (preserves ~70/15/15 ratio)
mermaid_cap_train = int(MERMAID_CAP * 0.70)
mermaid_cap_val = int(MERMAID_CAP * 0.15)
mermaid_cap_test = MERMAID_CAP - mermaid_cap_train - mermaid_cap_val
mermaid_train = _subsample(mermaid_train, mermaid_cap_train, sub_generator)
mermaid_val = _subsample(mermaid_val, mermaid_cap_val, sub_generator)
mermaid_test = _subsample(mermaid_test, mermaid_cap_test, sub_generator)

# Subsample CoralNet within source-split datasets proportionally
coralnet_cap_train = int(CORALNET_CAP * 0.70)
coralnet_cap_val = int(CORALNET_CAP * 0.15)
coralnet_cap_test = CORALNET_CAP - coralnet_cap_train - coralnet_cap_val
coralnet_train = _subsample(coralnet_train, coralnet_cap_train, sub_generator)
coralnet_val = _subsample(coralnet_val, coralnet_cap_val, sub_generator)
coralnet_test = _subsample(coralnet_test, coralnet_cap_test, sub_generator)

print(f"MERMAID  (capped): train={len(mermaid_train)}, val={len(mermaid_val)}, test={len(mermaid_test)}")
print(f"CoralNet (capped): train={len(coralnet_train)}, val={len(coralnet_val)}, test={len(coralnet_test)}")

In [ ]:
combined_train = CombinedCoralDataset([mermaid_train, coralnet_train])
combined_val = CombinedCoralDataset([mermaid_val, coralnet_val])
combined_test = CombinedCoralDataset([mermaid_test, coralnet_test])

print(f"Combined splits: train={len(combined_train)}, val={len(combined_val)}, test={len(combined_test)}")

In [ ]:
train_loader = DataLoader(
    combined_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1,
    drop_last=True,
    collate_fn=combined_train.collate_fn,
)
val_loader = DataLoader(
    combined_val,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
    drop_last=True,
    collate_fn=combined_val.collate_fn,
)
test_loader = DataLoader(
    combined_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
    drop_last=True,
    collate_fn=combined_test.collate_fn,
)

In [ ]:
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

In [ ]:
combined_train.num_classes

# 3. Model

In [ ]:
meta_model = MetaModel(
    run_name=cfg.run_name,
    num_classes=combined_train.num_classes,
    device=device,
    model_kwargs=cfg.model,
    training_kwargs=cfg.training,
)

evaluator = EvaluatorSemanticSegmentation(
    num_classes=combined_train.num_classes,
    device=device,
)

In [ ]:
cfg.logger.experiment_name = "mermaid_combined"
cfg_logger.logger.experiment_name = "mermaid_combined"

In [ ]:
logger = Logger(
    config=cfg_logger,
    meta_model=meta_model,
    log_epochs=cfg.logger.log_epochs,
    log_checkpoint=cfg.logger.get("log_checkpoint", 5),
    checkpoint_dir=".",
    enable_mlflow=True,
    id2label=combined_train.id2label,
)

In [ ]:
run = mlflow.active_run()
assert run is not None, "MLflow run was not started — check MLFLOW_TRACKING_URI and WARNING logs above"
print(f"MLflow run_id: {run.info.run_id}")

In [ ]:
tracking_uri = mlflow.get_tracking_uri()
print(f"Tracking URI: {tracking_uri}")

In [ ]:
logger.log_datasets(combined_train, context="training")

mlflow.log_params(
    {
        "data/mermaid_total_size": total_size,
        "data/mermaid_train_size": train_size,
        "data/mermaid_val_size": val_size,
        "data/mermaid_test_size": test_size,
        "data/combined_train_size": len(combined_train),
        "data/combined_val_size": len(combined_val),
        "data/combined_test_size": len(combined_test),
        "data/seed": seed,
    }
)

In [ ]:
train_model(meta_model, evaluator, train_loader, val_loader, logger=logger)

## Reconnect After Window Close

The kernel keeps running after the browser disconnects. Re-run all setup cells, then use the `run_id` printed above to resume:

In [ ]:
# Paste run_id from the cell above
# active_run = resume_run("paste-run-id-here")